# 1. Import the necessary libraries

In [ ]:
from mesh.core import *
from mesh.parameters import MeshParameters
from microgrid_old.techno_ka import techno_ka
from problems import get_problem

from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from pickle import dump
from functools import partial

import numpy as np

# 2. Define the general configuration of the experiments

In [ ]:
# Execution configuration
num_runs = 30 # Number of runs
max_fitness_eval = 15000 # Maximum fitness evaluations (not used if it is less than one)
population_size = 100 # Population size
random_state = None # Defines a seed for random numbers (not used if it is None)
workers = 24

# Objective function configuration
experiments = ['zdt1', 'zdt2', 'zdt3', 'zdt4', 'zdt6']
objective_dim = 2 # Number of objectives
position_dim = 10 # Design space dimension

## 2.1 Objective function of the Microgrid

In [ ]:
# Microgrid objetive function
# position_min_value = np.array([10, 1, 50]) # Lower bound of problem [max PV generation, number of wind turbines, battery capacity]
# position_max_value = np.array([450, 5, 500]) # Upper bound of problem [max PV generation, number of wind turbines, battery capacity]
# def func(args):
#     r = techno_ka(args[0], args[1], 0.8, args[2], select_bat, solar_data, wind_data, load_ind)[:objective_dim]
#     #r = techno_ka(args[0], args[1], 0.8, args[2], select_bat, solar_data, wind_data, load_ind)[1:3]
#     r[-1] = -r[-1] # Maximizing renewable factor
#     return r

# 3. Define functions to run experiments in parallel with processes

In [ ]:
def execute_with_parallelism(func, params_list, max_workers=4):
	results = []
	with ProcessPoolExecutor(max_workers=max_workers) as executor:
		# Send tasks for execution (a dict)
		future_to_params = {executor.submit(func, *params): params for params in params_list}
		# Collect results as tasks are completed
		for future in as_completed(future_to_params):
			params = future_to_params[future]
			try:
				result = future.result()
				print(f"Result for {params}: {result}")
				results.append((params, result))
			except Exception as e:
				print(f"Error when execute with {params}: {e}")
				results.append((params, None))
	return results

def list_of_problems(func_name, position_dim, objective_dim):
	func, position_min_value, position_max_value = get_problem(func_name, n_var=position_dim, n_obj=objective_dim)
	return func, position_min_value, position_max_value

# 4. Run experiments with the algorithms

## 4.1 Run vectorized MESH in the experiments (in parallel)